# Task 2: Consuming a RESTful API with a Frontend Interface

**Goal:** Build a simple Python frontend using **Streamlit** that consumes a public RESTful API and displays the retrieved data.

In this notebook, we will use the free JSONPlaceholder API.  
It gives sample data such as users, posts, comments, albums, and todos.

**Main tools used:**
- `requests` for calling the RESTful API
- `streamlit` for creating the frontend interface
- `pandas` for showing data in table form

In [ ]:
# Install required libraries.
# Run this cell once before running the app.
!pip install streamlit requests pandas

## Step 1: Test the RESTful API in the notebook

Before creating the frontend, first check that the API is working.

In [ ]:
import requests
import pandas as pd

# Base URL of the public RESTful API
BASE_URL = "https://jsonplaceholder.typicode.com"

# We will fetch posts from the API
response = requests.get(f"{BASE_URL}/posts")

# Check if request was successful
print("Status Code:", response.status_code)

# Convert JSON response into Python list/dictionary
data = response.json()

# Display first 3 records
data[:3]

In [ ]:
# Convert retrieved JSON data into a Pandas DataFrame
df = pd.DataFrame(data)

# Show first 10 rows
df.head(10)

## Step 2: Create Streamlit frontend app

This cell creates a file named `api_frontend_app.py`.  
The app will allow the user to select an API endpoint and view the retrieved data.

In [ ]:
%%writefile api_frontend_app.py
import streamlit as st
import requests
import pandas as pd

# -----------------------------
# Frontend title and description
# -----------------------------
st.set_page_config(page_title="REST API Consumer", layout="wide")
st.title("RESTful API Consumer Frontend")
st.write("This app consumes data from JSONPlaceholder RESTful API and displays it in a simple frontend.")

# -----------------------------
# API configuration
# -----------------------------
BASE_URL = "https://jsonplaceholder.typicode.com"

# Endpoint options for the user
endpoint = st.selectbox(
    "Select API endpoint:",
    ["posts", "users", "comments", "albums", "todos"]
)

# Button to fetch data
fetch_button = st.button("Fetch Data from API")

# -----------------------------
# Function to call RESTful API
# -----------------------------
def fetch_api_data(selected_endpoint):
    """
    This function sends a GET request to the selected API endpoint.
    It returns JSON data if the API request is successful.
    """
    try:
        url = f"{BASE_URL}/{selected_endpoint}"
        response = requests.get(url, timeout=10)

        # Raise error if status code is not successful
        response.raise_for_status()

        return response.json(), None

    except requests.exceptions.RequestException as error:
        return None, str(error)

# -----------------------------
# Display retrieved API data
# -----------------------------
if fetch_button:
    data, error = fetch_api_data(endpoint)

    if error:
        st.error(f"API request failed: {error}")
    else:
        st.success(f"Data retrieved successfully from /{endpoint}")

        # Convert JSON data to DataFrame for clean display
        df = pd.DataFrame(data)

        st.subheader("Data Table")
        st.dataframe(df, use_container_width=True)

        # Search option
        st.subheader("Search in Retrieved Data")
        search_text = st.text_input("Enter text to search:")

        if search_text:
            # Convert all columns to string and search in each row
            filtered_df = df[df.astype(str).apply(
                lambda row: row.str.contains(search_text, case=False, na=False).any(),
                axis=1
            )]

            st.write(f"Search Results: {len(filtered_df)} record(s) found")
            st.dataframe(filtered_df, use_container_width=True)

        # Show raw JSON also
        with st.expander("Show Raw JSON"):
            st.json(data[:5] if isinstance(data, list) else data)

# -----------------------------
# Footer
# -----------------------------
st.info("Example API used: JSONPlaceholder. This is a free fake REST API for testing and practice.")

## Step 3: Run the frontend app

For local Jupyter Notebook / Anaconda, run the next command and open the shown local URL.

For Google Colab, Streamlit may need tunneling such as `localtunnel` or `ngrok`.

In [ ]:
# Run this command in notebook or terminal.
# The app will open at http://localhost:8501
!streamlit run api_frontend_app.py

## Expected Output

The frontend should show:
- a dropdown to select API endpoint
- a button to fetch data
- retrieved API data in table form
- a search box
- raw JSON preview

This completes the API consuming frontend task.